# Apache Polaris — Rotate Principal (Live Test Harness)

## Concept
`polaris_rotate_principal.ipynb` takes an existing `PRINCIPAL_NAME` as a
parameter — it's a targeted utility, not a test generator, so it doesn't
create fixtures itself. This notebook is the companion that does: it
**provisions a throwaway test principal with real role wiring** (a
principal-role, a dedicated catalog, a catalog-role, and an actual
`CATALOG_MANAGE_CONTENT` grant), runs it through the identical 8-step
rotate flow, then **tears everything down** — proving live, end-to-end,
that rotation via `/reset` + `/rotate` leaves role wiring and authorization
intact, using disposable data instead of risking a real principal.

## Flow
```
1.  Provision   principal + principal-role + catalog + catalog-role + grant
2.  Baseline    snapshot principal/role/grant state BEFORE rotation
3.  NEGATIVE    root /rotate on the test principal            → 403
4.  NEGATIVE    grant PRINCIPAL_ROTATE_CREDENTIALS to service_admin → 404
5.  NEGATIVE    impersonation ruled out (see polaris_rotate_principal.ipynb
                Step 4 for the full writeup — condensed here)
6.  /reset      root resets the test principal's credentials    → 200
7.  VERIFY      principal, principal-role, catalog-role, grant all unchanged
8.  /rotate     test principal self-rotates using the reset creds → 200
9.  VERIFY      final creds authenticate AND can still list namespaces on
                the test catalog (proves the grant, not just the login,
                survived rotation) — AND that the Step-6 creds still work
                too (expected — see below)
10. VERIFY      a SECOND rotation finally evicts the Step-6 creds
11. Teardown    delete catalog-role → principal-role → principal → catalog
12. Sweep       backstop: remove any `rotate-test-*` leftovers from a
                previous crashed run, so repeated runs never accumulate
```

## ⚠️ Credentials don't die after one rotation — Polaris keeps two secret slots
Measured on a live run (2026-07-08), not assumed: Polaris stores a **main**
and a **secondary** secret per principal, and accepts either as valid. A
plain `/rotate` only demotes the current main secret to secondary and mints
a new main — the demoted secret **keeps working** for exactly one more
generation. `/reset` forces two internal rotations in one call, so it fully
evicts whatever was active right before it, but a single `/rotate` does not.
Step 9 below asserts the Step-6 credentials are still alive (expected, not
a leak); Step 10 rotates again and confirms they're finally dead.

Steps 3–9 mirror `polaris_rotate_principal.ipynb` cell-for-cell — see that
notebook for the full authorization writeup on each negative result. This
notebook exists to prove those same steps against **real, disposable**
role wiring rather than asking you to point it at a principal you care
about.

## Safety — LOCAL DEVICE ONLY
Same hardcoded-local convention as every notebook in `admin/`. This one
both **creates and deletes** real entities. No `PRINCIPAL_NAME` parameter
to set — a unique test principal name (`rotate-test-<unix-timestamp>`) is
generated each run, so re-running never collides with a previous run.
The teardown (step 10) and sweep (step 11) cells are safe to re-run any
number of times (deletes tolerate 404) — if a run fails partway, just run
the rest of the notebook (or just the sweep cell) to clean up.

## How to run
Restart & Run All. No parameters to set.


In [32]:
# ============================================================
# Prerequisites:  uv add requests
# ============================================================
import subprocess, sys, os

REQUIRED = ['requests']

def install(pkg):
    result = subprocess.run(['uv', 'add', pkg], capture_output=True, text=True, cwd=os.getcwd())
    if result.returncode == 0:
        return '✅', 'uv'
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], capture_output=True, text=True)
    return ('✅', 'pip') if result.returncode == 0 else ('❌', 'failed')

for pkg in REQUIRED:
    status, method = install(pkg)
    print(f'  {status} {pkg} ({method})')
print('\n⚠️  Restart kernel after first install.')

  ✅ requests (uv)

⚠️  Restart kernel after first install.


In [33]:
# ============================================================
# Setup — hardcoded local device only (see Safety note above)
# ============================================================
import sys, pathlib, time
from IPython.display import display

_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / "src"))
from polaris_rest import PolarisREST

POLARIS_URL     = 'http://192.168.139.2:8181'
REALM           = 'POLARIS'
ROOT            = 'root'
ROOT_SECRET     = 'polaris-secret'
BUCKET          = 'data-catalog-bucket'
MINIO_ENDPOINT  = 'http://192.168.139.2:9000'

# ── SAFETY GUARD: LOCAL DEVICE ONLY ──────────────────────────────────
assert any(t in POLARIS_URL for t in ("127.0.0.1", "localhost", "192.168.139.2")), \
    f"rotate_principal_test is LOCAL ONLY — refusing non-local host: {POLARIS_URL}"

pc = PolarisREST(POLARIS_URL, REALM)
r = pc.get_token(ROOT, ROOT_SECRET)
pc.token = r.json()['access_token']
print('✅ Connected as root')

✅ Connected as root


In [34]:
# ============================================================
# Parameters — auto-generated, unique per run (no manual input needed)
# ============================================================
TEST_PREFIX    = 'rotate-test-'
TEST_PRINCIPAL = f'{TEST_PREFIX}{int(time.time())}'
TEST_ROLE      = f'{TEST_PRINCIPAL}_role'
TEST_CATALOG   = f'{TEST_PRINCIPAL}_catalog'
TEST_CR        = f'{TEST_PRINCIPAL}_cr'

print(f'TEST_PRINCIPAL = {TEST_PRINCIPAL}')
print(f'TEST_ROLE      = {TEST_ROLE}')
print(f'TEST_CATALOG   = {TEST_CATALOG}')
print(f'TEST_CR        = {TEST_CR}')

TEST_PRINCIPAL = rotate-test-1783476047
TEST_ROLE      = rotate-test-1783476047_role
TEST_CATALOG   = rotate-test-1783476047_catalog
TEST_CR        = rotate-test-1783476047_cr


In [35]:
# ============================================================
# Step 1 — Provision a throwaway test principal WITH real role wiring
# ============================================================
r = pc.create_catalog(TEST_CATALOG, bucket=BUCKET, minio_endpoint=MINIO_ENDPOINT)
assert r.status_code in (200, 201), f'create_catalog failed: {r.status_code} {r.text}'
print(f'✅ catalog: {TEST_CATALOG}')

r = pc.create_principal(TEST_PRINCIPAL)
assert r.status_code in (200, 201), f'create_principal failed: {r.status_code} {r.text}'
print(f'✅ principal: {TEST_PRINCIPAL}')

r = pc.create_principal_role(TEST_ROLE)
assert r.status_code in (200, 201), f'create_principal_role failed: {r.status_code} {r.text}'
print(f'✅ principal-role: {TEST_ROLE}')

r = pc.assign_principal_role_to_principal(TEST_PRINCIPAL, TEST_ROLE)
assert r.status_code in (200, 201), f'assign principal-role failed: {r.status_code} {r.text}'
print(f'✅ assigned {TEST_ROLE} → {TEST_PRINCIPAL}')

r = pc.create_catalog_role(TEST_CATALOG, TEST_CR)
assert r.status_code in (200, 201), f'create_catalog_role failed: {r.status_code} {r.text}'
print(f'✅ catalog-role: {TEST_CR}')

r = pc.assign_catalog_role_to_principal_role(TEST_CATALOG, TEST_ROLE, TEST_CR)
assert r.status_code in (200, 201), f'assign catalog-role failed: {r.status_code} {r.text}'
print(f'✅ assigned {TEST_CR} → {TEST_ROLE}')

r = pc.grant_privilege(TEST_CATALOG, TEST_CR, 'CATALOG_MANAGE_CONTENT')
assert r.status_code in (200, 201), f'grant_privilege failed: {r.status_code} {r.text}'
print(f'✅ granted CATALOG_MANAGE_CONTENT on {TEST_CATALOG} to {TEST_CR}')

print('\n✅ Test footprint provisioned.')

✅ catalog: rotate-test-1783476047_catalog
✅ principal: rotate-test-1783476047
✅ principal-role: rotate-test-1783476047_role
✅ assigned rotate-test-1783476047_role → rotate-test-1783476047
✅ catalog-role: rotate-test-1783476047_cr
✅ assigned rotate-test-1783476047_cr → rotate-test-1783476047_role
✅ granted CATALOG_MANAGE_CONTENT on rotate-test-1783476047_catalog to rotate-test-1783476047_cr

✅ Test footprint provisioned.


In [36]:
# ============================================================
# Step 2 — Baseline snapshot (BEFORE rotation)
# ============================================================
r = pc.get_principal(TEST_PRINCIPAL)
assert r.status_code == 200
print(f'✅ principal exists: {TEST_PRINCIPAL}')

r = pc.get_principal_role(TEST_ROLE)
assert r.status_code == 200
print(f'✅ principal-role exists: {TEST_ROLE}')

r = pc.list_principals_for_principal_role(TEST_ROLE)
holders_before = sorted(p['name'] for p in r.json().get('principals', []))
assert holders_before == [TEST_PRINCIPAL]
print(f'✅ role holders: {holders_before}')

r = pc.list_catalog_roles(TEST_CATALOG)
cr_names_before = sorted(cr['name'] for cr in r.json().get('roles', []))
assert TEST_CR in cr_names_before
print(f'✅ catalog-roles on {TEST_CATALOG}: {cr_names_before}')

r = pc.list_grants(TEST_CATALOG, TEST_CR)
grants_before = sorted(g['privilege'] for g in r.json().get('grants', []))
assert 'CATALOG_MANAGE_CONTENT' in grants_before
print(f'✅ grants on {TEST_CR}: {grants_before}')

✅ principal exists: rotate-test-1783476047
✅ principal-role exists: rotate-test-1783476047_role
✅ role holders: ['rotate-test-1783476047']
✅ catalog-roles on rotate-test-1783476047_catalog: ['catalog_admin', 'rotate-test-1783476047_cr']
✅ grants on rotate-test-1783476047_cr: ['CATALOG_MANAGE_CONTENT']


In [37]:
# ============================================================
# Step 3 — NEGATIVE RESULT: root attempts the self-service /rotate
# ============================================================
r = pc.rotate_credentials(TEST_PRINCIPAL)  # pc.token = root
print(f'Status: {r.status_code}')
print(r.text)

assert r.status_code == 403, f'Expected 403 (self-only by design) — got {r.status_code}.'
print('\n✅ Confirmed: /rotate is self-service only, same as the real-principal notebook.')

Status: 403
{"error":{"message":"Principal 'root' with activated PrincipalRoles '[service_admin]' and activated grants via '[service_admin]' is not authorized for op ROTATE_CREDENTIALS","type":"ForbiddenException","code":403}}

✅ Confirmed: /rotate is self-service only, same as the real-principal notebook.


In [38]:
# ============================================================
# Step 4 — NEGATIVE RESULT: attempt to grant the missing privilege
# ============================================================
import requests

r = requests.put(
    f'{pc.base_mgmt}/principal-roles/service_admin/grants',
    headers=pc._h(),
    json={'grant': {'type': 'principal', 'privilege': 'PRINCIPAL_ROTATE_CREDENTIALS'}},
)
print(f'Status: {r.status_code}')
print(r.text)

assert r.status_code == 404, f'Expected 404 (no such grantable resource) — got {r.status_code}.'
print('\n✅ Confirmed: no privilege exists to close this gap (Principal is not a securable object).')

Status: 404
{"error":{"message":"Unable to find matching target resource method","type":"NotFoundException","code":404}}

✅ Confirmed: no privilege exists to close this gap (Principal is not a securable object).


In [39]:
# ============================================================
# Step 5 — Impersonation ruled out (condensed — see
# polaris_rotate_principal.ipynb Step 4 for the full writeup)
# ============================================================
print('ℹ️  Same conclusion as the real-principal notebook: Polaris\'s OAuth endpoint has no')
print('   token-exchange/act-as grant type, and root cannot read an existing principal\'s')
print('   secret to fake one. /reset (next cell) is the actual admin-callable path.')

ℹ️  Same conclusion as the real-principal notebook: Polaris's OAuth endpoint has no
   token-exchange/act-as grant type, and root cannot read an existing principal's
   secret to fake one. /reset (next cell) is the actual admin-callable path.


In [40]:
# ============================================================
# Step 6 — Root resets the test principal's credentials via /reset
# ============================================================
r = pc.reset_principal_credentials(TEST_PRINCIPAL)
print(f'Status: {r.status_code}')
assert r.status_code == 200, (
    f'Reset failed ({r.status_code}): {r.text}\n'
    "If this 404s, ENABLE_CREDENTIAL_RESET may be disabled on this server."
)

creds = r.json().get('credentials', {})
CLIENT_ID     = creds.get('clientId')
CLIENT_SECRET = creds.get('clientSecret')
print(f'✅ New credentials for {TEST_PRINCIPAL} (root-issued reset):')
print(f'   clientId:     {CLIENT_ID}')
print(f'   clientSecret: {CLIENT_SECRET}')

Status: 200
✅ New credentials for rotate-test-1783476047 (root-issued reset):
   clientId:     0c8d1a9c7db73993
   clientSecret: 973a860bb87546bfe61d5666aeb3f4ea


In [41]:
# ============================================================
# Step 7 — VERIFY wiring + grants survived the reset
# ============================================================
r = pc.get_principal(TEST_PRINCIPAL)
assert r.status_code == 200, 'Principal disappeared after reset?!'
print(f'✅ principal still exists: {TEST_PRINCIPAL}')

r = pc.list_principals_for_principal_role(TEST_ROLE)
holders_after = sorted(p['name'] for p in r.json().get('principals', []))
assert holders_after == holders_before, f'Role assignment changed: {holders_before} -> {holders_after}'
print(f'✅ role assignment unchanged: {holders_after}')

r = pc.list_catalog_roles(TEST_CATALOG)
cr_names_after = sorted(cr['name'] for cr in r.json().get('roles', []))
assert cr_names_after == cr_names_before
print(f'✅ catalog-roles unchanged: {cr_names_after}')

r = pc.list_grants(TEST_CATALOG, TEST_CR)
grants_after = sorted(g['privilege'] for g in r.json().get('grants', []))
assert grants_after == grants_before, f'Grants changed: {grants_before} -> {grants_after}'
print(f'✅ grants unchanged: {grants_after}')

print('\n✅ Reset preserved every bit of role wiring and every grant.')

✅ principal still exists: rotate-test-1783476047
✅ role assignment unchanged: ['rotate-test-1783476047']
✅ catalog-roles unchanged: ['catalog_admin', 'rotate-test-1783476047_cr']
✅ grants unchanged: ['CATALOG_MANAGE_CONTENT']

✅ Reset preserved every bit of role wiring and every grant.


In [42]:
# ============================================================
# Step 8 — Self-rotate demo: the test principal rotates its OWN new credentials
# ============================================================
r = pc.get_token(CLIENT_ID, CLIENT_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r.status_code == 200, f'Could not authenticate with reset creds: {r.text}'
principal_token = r.json()['access_token']
print(f'✅ authenticated as {TEST_PRINCIPAL} using the reset credentials')

r = pc.rotate_credentials(TEST_PRINCIPAL, token=principal_token)
print(f'Status: {r.status_code}')
assert r.status_code == 200, f'Self-rotate failed ({r.status_code}): {r.text}'

creds2 = r.json().get('credentials', {})
FINAL_CLIENT_ID     = creds2.get('clientId')
FINAL_CLIENT_SECRET = creds2.get('clientSecret')
print(f'✅ self-rotated successfully: clientId={FINAL_CLIENT_ID}')

✅ authenticated as rotate-test-1783476047 using the reset credentials
Status: 200
✅ self-rotated successfully: clientId=0c8d1a9c7db73993


In [43]:
# ============================================================
# Step 9 — VERIFY final creds work end-to-end (auth AND still authorized)
# ============================================================
r = pc.get_token(FINAL_CLIENT_ID, FINAL_CLIENT_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r.status_code == 200, f'Final credentials do not work: {r.status_code} {r.text}'
final_token = r.json()['access_token']
print(f'✅ final credentials authenticate successfully as {TEST_PRINCIPAL}')

# prove the CATALOG_MANAGE_CONTENT grant survived, not just the login
r = pc.list_namespaces(TEST_CATALOG, token=final_token)
assert r.status_code == 200, f'Rotated principal lost its grant: {r.status_code} {r.text}'
print(f'✅ rotated principal can still list namespaces on {TEST_CATALOG} — grant survived rotation')

# ⚠️ MEASURED LIVE (2026-07-08): the Step 6 (/reset) credentials do NOT die
# here. Polaris stores two secret slots per principal (main + secondary);
# a plain /rotate only demotes the previous main secret to secondary — it
# stays valid for exactly one more generation. Full eviction takes a SECOND
# rotation (see the next cell). So the correct assertion is that the Step-6
# credentials STILL work:
r_step6 = pc.get_token(CLIENT_ID, CLIENT_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_step6.status_code == 200, (
    "Expected the Step-6 (/reset) credentials to still authenticate as the "
    f"'secondary' secret after a single subsequent /rotate — got {r_step6.status_code}."
)
print(f"✅ Step-6 credentials still work ({r_step6.status_code}) — they're now the "
      "'secondary' secret, by design. Not a leak — see the next cell.")

✅ final credentials authenticate successfully as rotate-test-1783476047
✅ rotated principal can still list namespaces on rotate-test-1783476047_catalog — grant survived rotation
✅ Step-6 credentials still work (200) — they're now the 'secondary' secret, by design. Not a leak — see the next cell.


In [44]:
# ============================================================
# Step 10 — Prove full eviction requires a SECOND rotation
# ============================================================
r = pc.get_token(FINAL_CLIENT_ID, FINAL_CLIENT_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r.status_code == 200
current_token = r.json()['access_token']

r = pc.rotate_credentials(TEST_PRINCIPAL, token=current_token)
assert r.status_code == 200, f'Second rotate failed: {r.status_code} {r.text}'
creds3 = r.json()['credentials']
print(f"✅ rotated again: clientId={creds3['clientId']}")

r_step6_final = pc.get_token(CLIENT_ID, CLIENT_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_step6_final.status_code != 200, (
    f'Step-6 credentials STILL work after two rotations ({r_step6_final.status_code}) — '
    'expected full eviction by now.'
)
print(f'✅ Step-6 credentials are finally dead ({r_step6_final.status_code}) — confirms the '
      'two-rotation eviction rule live, not just from source-reading.')

✅ rotated again: clientId=0c8d1a9c7db73993
✅ Step-6 credentials are finally dead (401) — confirms the two-rotation eviction rule live, not just from source-reading.


In [45]:
# ============================================================
# Step 10 — Teardown (idempotent — safe to re-run)
# ============================================================
r = pc.delete_catalog_role(TEST_CATALOG, TEST_CR)
print(f"{'✅' if r.status_code in (200,204,404) else '⚠️ '} [{r.status_code}] catalog-role: {TEST_CR}")

r = pc.delete_principal_role(TEST_ROLE)
print(f"{'✅' if r.status_code in (200,204,404) else '⚠️ '} [{r.status_code}] principal-role: {TEST_ROLE}")

r = pc.delete_principal(TEST_PRINCIPAL)
print(f"{'✅' if r.status_code in (200,204,404) else '⚠️ '} [{r.status_code}] principal: {TEST_PRINCIPAL}")

r = pc.delete_catalog(TEST_CATALOG, purge=True)
print(f"{'✅' if r.status_code in (200,204,404) else '⚠️ '} [{r.status_code}] catalog: {TEST_CATALOG}")

# verify
assert pc.get_principal(TEST_PRINCIPAL).status_code == 404
assert pc.get_principal_role(TEST_ROLE).status_code == 404
assert pc.get_catalog(TEST_CATALOG).status_code == 404
print('\n✅ Teardown verified — no trace of this run left behind.')

✅ [204] catalog-role: rotate-test-1783476047_cr
✅ [204] principal-role: rotate-test-1783476047_role
✅ [204] principal: rotate-test-1783476047
✅ [204] catalog: rotate-test-1783476047_catalog

✅ Teardown verified — no trace of this run left behind.


In [46]:
# ============================================================
# Step 11 — Safety sweep (backstop for a previous crashed run)
# ============================================================
r = pc.list_principals()
leftover_principals = [
    p['name'] for p in r.json().get('principals', [])
    if p['name'].startswith(TEST_PREFIX)
]
for name in leftover_principals:
    pc.delete_principal(name)
    pc.delete_principal_role(f'{name}_role')
    pc.delete_catalog(f'{name}_catalog', purge=True)
    print(f'🧹 swept leftover: {name}')

if not leftover_principals:
    print('✅ nothing to sweep — no leftover rotate-test-* principals found.')

🧹 swept leftover: rotate-test-1783475302
